In [33]:
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          DataCollatorWithPadding,
                          TrainingArguments,
                          pipeline
                        )

/home/abdullah/python_virtual_env/gradio_env/lib/python3.8/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/abdullah/python_virtual_env/gradio_env/lib/python3.8/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


## Load Dataset

In [41]:
data_path = "../data/jutsus.jsonl"

In [42]:
df = pd.read_json(data_path, lines=True)
df.head()

,jutsu_name,jutsu_type,jutsu_description
0,All Weapons Above Heaven,Ninjutsu,This technique raises all the status boosts (S...
1,Air Raid Shot,Ninjutsu,"Kankurō's puppet, Karasu, soars into the air w..."
2,Akuta,"Ninjutsu, Kinjutsu, Hiden",Akuta is an Earth Release technique that's cre...
3,Air Lightning Bullet,"Taijutsu, Shurikenjutsu",The user punches the opponent twice with their...
4,Air Gold Dust Protective Wall,"Kekkei Genkai, Ninjutsu","Making use of his Gold Dust, the Fourth Kazeka..."


In [43]:
def simplify_justu(jutsu):
    if 'Genjutsu' in jutsu:
        return 'Genjutsu'
    if 'Taijutsu' in jutsu:
        return 'Taijutsu'
    if 'Ninjutsu' in jutsu:
        return 'Ninjutsu'
    return None

In [44]:
df['jutsu_type_simplified'] = df['jutsu_type'].apply(simplify_justu)

In [45]:
df['jutsu_type_simplified'].value_counts()  

jutsu_type_simplified
Ninjutsu    1860
Taijutsu     580
Genjutsu      93
Name: count, dtype: int64

In [6]:
df['text'] = df['jutsu_name']+'. '+df['jutsu_description']
df['jutsu'] = df['jutsu_type_simplified']
df= df[['text','jutsu']]
df = df.dropna()

In [11]:
from bs4 import BeautifulSoup

class Cleaner():
  def __init__(self):
    pass
  def put_line_breaks(self,text):
    text = text.replace('</p>','</p>\n')
    return text
  def remove_html_tags(self,text):
    cleantext = BeautifulSoup(text, "lxml").text
    return cleantext
  def clean(self,text):
    text = self.put_line_breaks(text)
    text = self.remove_html_tags(text)
    return text

In [15]:
text_column_name = 'text'
label_column_name = 'jutsu'

In [13]:
# Clean Text
cleaner = Cleaner()
df['text_cleaned'] = df[text_column_name].apply(cleaner.clean)

/tmp/ipykernel_19068/1671959954.py:10: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  cleantext = BeautifulSoup(text, "lxml").text


In [18]:
# Encode Labels
le = preprocessing.LabelEncoder()
le.fit(df[label_column_name].tolist())

LabelEncoder()

In [19]:
label_dict = { index:label_name for index, label_name in  enumerate(le.__dict__['classes_'].tolist())}
label_dict = label_dict
df['label'] = le.transform(df[label_column_name].tolist())

In [21]:
test_size = 0.2

In [22]:
# Train/Test Split
df_train,df_test = train_test_split(df,
                                    test_size=test_size,
                                    stratify=df['label'])

In [23]:
df_train.shape, df_test.shape

((2026, 4), (507, 4))

In [25]:
def preprocess_function(tokenizer, examples):
    return tokenizer(examples["text_cleaned"], truncation=True)

In [28]:
model_name = "distilbert-base-uncased"

In [29]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/abdullah/python_virtual_env/gradio_env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [31]:
# Convert to Huggingface Dataset
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

# Tokenize
tokenized_train = train_dataset.map(lambda x: preprocess_function(tokenizer,x)
                                , batched=True)

tokenized_test = test_dataset.map(lambda x: preprocess_function(tokenizer,x)
                                , batched=True)

Map:   0%|          | 0/2026 [00:00<?, ? examples/s]

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

## Load model

In [34]:
model = pipeline("text-classification", model="AbdullahTarek/jutsu_classifier", return_all_scores=True)

/home/abdullah/python_virtual_env/gradio_env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/home/abdullah/python_virtual_env/gradio_env/lib/python3.8/site-packages/transformers/pipelines/text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [35]:
text = """ the Rasengan consists of concentrating and rotating the chakra at a focal point on the user's hand. The result is a spinning chakra sphere with immense destructive power. Unlike the Chidori, which has a more external impact, the Rasengan can reach deep into a target. """

In [36]:
model_output = model(text)

In [37]:
model_output

[[{'label': 'Genjutsu', 'score': 0.034382451325654984},
  {'label': 'Ninjutsu', 'score': 0.9222820401191711},
  {'label': 'Taijutsu', 'score': 0.04333556070923805}]]

In [38]:
def postprocess(model_output):
    output=[]
    for pred in model_output:
        label = max(pred, key=lambda x: x['score'])['label']
        output.append(label)
    return output

In [39]:
postprocess(model_output)

['Ninjutsu']